In [24]:
# Cell 0: Imports and data loading
import pandas as pd
import numpy as np
from numpy.linalg import lstsq
from collections import defaultdict
from sklearn.metrics import accuracy_score

# news
news = pd.read_csv('data/news_classified_final.csv')
news['Date'] = pd.to_datetime(news['Date'], utc=True).dt.tz_localize(None).dt.normalize()
news['categories'] = news['categories'].apply(eval)

TICKERS = ['NVDA', 'MSFT', 'AMZN', 'META', 'GOOGL']

# per-stock label tables from individual CSVs
stock_labels = {}
for ticker in TICKERS:
    df = pd.read_csv(f'data/{ticker}.csv')
    df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None).dt.normalize()
    df = df.sort_values('Date').reset_index(drop=True)
    df['label'] = (df['Daily_Return'] > 0).astype(int)
    stock_labels[ticker] = df[['Date', 'Daily_Return', 'label']]

print(f"News rows: {len(news)}")
print(f"Stock label counts: { {t: len(stock_labels[t]) for t in TICKERS} }")

News rows: 13106
Stock label counts: {'NVDA': 1254, 'MSFT': 1254, 'AMZN': 1254, 'META': 1254, 'GOOGL': 1254}


In [31]:
# Cell 1: LP function
def run_lp(news_df, label_df, train_start, train_end, 
           test_start='2025-01-01',
           min_test_samples=10,
           filter_zero_sentiment=True
           ):
    df = pd.merge(news_df, label_df, on='Date', how='inner')
    df = df.sort_values('Date')
    df['next_return'] = df['Daily_Return'].shift(-1)
    df['next_label'] = (df['next_return'] > 0).astype(int)
    df = df.dropna(subset=['next_return', 'sentiment'])

    train = df[(df['Date'] >= train_start) & (df['Date'] < train_end)]
    test  = df[df['Date'] >= test_start].copy()

    if filter_zero_sentiment:
        train = train[train['sentiment'] != 0]
        test  = test[test['sentiment'] != 0]
        
    if len(train) < 15 or len(test) < min_test_samples:
        return None

    X = np.column_stack([train['sentiment'].values, train['Daily_Return'].values, np.ones(len(train))])
    y = train['next_return'].values
    coef, _, _, _ = lstsq(X, y, rcond=None)
    beta = coef[0]

    test['pred'] = (beta * test['sentiment'] > 0).astype(int)
    acc      = accuracy_score(test['next_label'], test['pred'])
    baseline = test['next_label'].mean()

    return {'beta': beta, 'acc': acc, 'baseline': baseline, 'n': len(test)}

print("run_lp ready.")

run_lp ready.


In [32]:
# Cell 2: Aggregate daily sentiment by ticker x category
CATEGORIES = ['algorithm', 'chip', 'power', 'regulation', 'earnings']

daily_sentiment = {}  # key: (ticker, cat)

for ticker in TICKERS:
    for cat in CATEGORIES:
        subset = news[
            (news['ticker'] == ticker) &
            (news['categories'].apply(lambda x: cat in x))
        ]
        if len(subset) == 0:
            continue
        daily = subset.groupby('Date')['finbert_score'].sum().reset_index()
        daily.columns = ['Date', 'sentiment']
        daily_sentiment[(ticker, cat)] = daily

print(f"Total (ticker, cat) combinations: {len(daily_sentiment)}")

Total (ticker, cat) combinations: 25


In [33]:
# Cell 3: Run LP for all ticker x category combinations
results = []

for (ticker, cat), daily in daily_sentiment.items():
    label_df = stock_labels[ticker]
    for train_start, train_end, label in [
        ('2021-01-01', '2025-01-01', '2021-2024'),
        ('2023-01-01', '2025-01-01', '2023-2024')
    ]:
        res = run_lp(daily, label_df, train_start, train_end)
        if res is None:
            continue
        results.append({
            'ticker': ticker,
            'cat': cat,
            'train': label,
            **res
        })

results_df = pd.DataFrame(results)

In [34]:
results_df['beat'] = (results_df['acc'] > results_df['baseline']) & (results_df['acc'] > 0.5)
valid = results_df[results_df['beat']].copy()
valid = valid.sort_values(['ticker', 'cat', 'train'], ascending=[True, True, True])

print(f"{'Ticker':<8} {'Cat':<12} {'Train':<12} {'Beta':>10} {'Acc':>8} {'Baseline':>10} {'N':>5} {'Beat'}")
print('-' * 70)

last_ticker = None
for _, row in valid.iterrows():
    if row['ticker'] != last_ticker:
        print()
        last_ticker = row['ticker']
    beat = 'Yes' if row['beat'] else 'No'
    print(f"{row['ticker']:<8} {row['cat']:<12} {row['train']:<12} {row['beta']:>+10.5f} {row['acc']:>8.1%} {row['baseline']:>10.1%} {row['n']:>5} {beat}")

Ticker   Cat          Train              Beta      Acc   Baseline     N Beat
----------------------------------------------------------------------

AMZN     power        2021-2024      +0.00689    58.1%      51.6%    62 Yes
AMZN     power        2023-2024      +0.00390    58.1%      51.6%    62 Yes
AMZN     regulation   2021-2024      -0.00007    71.4%      50.0%    14 Yes
AMZN     regulation   2023-2024      -0.00052    71.4%      50.0%    14 Yes

GOOGL    power        2021-2024      +0.01262    59.0%      52.5%    61 Yes
GOOGL    power        2023-2024      +0.01456    59.0%      52.5%    61 Yes
GOOGL    regulation   2021-2024      +0.00041    51.3%      43.6%    39 Yes
GOOGL    regulation   2023-2024      +0.00082    51.3%      43.6%    39 Yes

META     earnings     2021-2024      +0.01996    57.1%      52.4%    42 Yes
META     earnings     2023-2024      +0.00445    57.1%      52.4%    42 Yes

MSFT     chip         2021-2024      +0.00554    51.6%      45.2%    31 Yes
MSFT     chi

In [35]:
invalid = results_df[~results_df['beat']].copy()
invalid = invalid.sort_values(['ticker', 'cat', 'train'], ascending=[True, True, True])

print(f"{'Ticker':<8} {'Cat':<12} {'Train':<12} {'Beta':>10} {'Acc':>8} {'Baseline':>10} {'N':>5} {'Beat'}")
print('-' * 70)

last_ticker = None
for _, row in invalid.iterrows():
    if row['ticker'] != last_ticker:
        print()
        last_ticker = row['ticker']
    print(f"{row['ticker']:<8} {row['cat']:<12} {row['train']:<12} {row['beta']:>+10.5f} {row['acc']:>8.1%} {row['baseline']:>10.1%} {row['n']:>5} No")

Ticker   Cat          Train              Beta      Acc   Baseline     N Beat
----------------------------------------------------------------------

AMZN     algorithm    2021-2024      -0.00446    49.0%      42.9%    49 No
AMZN     algorithm    2023-2024      -0.00187    49.0%      42.9%    49 No
AMZN     chip         2021-2024      -0.00373    25.0%      62.5%    24 No
AMZN     chip         2023-2024      -0.00597    25.0%      62.5%    24 No
AMZN     earnings     2021-2024      +0.00735    42.9%      45.2%    42 No
AMZN     earnings     2023-2024      +0.00029    42.9%      45.2%    42 No

GOOGL    algorithm    2021-2024      +0.00209    51.4%      51.4%    74 No
GOOGL    algorithm    2023-2024      +0.00142    51.4%      51.4%    74 No
GOOGL    chip         2021-2024      +0.00349    38.7%      61.3%    31 No
GOOGL    chip         2023-2024      +0.00384    38.7%      61.3%    31 No
GOOGL    earnings     2021-2024      +0.01221    50.0%      65.0%    40 No
GOOGL    earnings     202

In [37]:
# ============================================================
# Build equal-weighted portfolio from stock_labels
# ============================================================

# Merge daily returns from all stocks
returns_df = None
for ticker in TICKERS:
    df = stock_labels[ticker][['Date', 'Daily_Return']].copy()
    df = df.rename(columns={'Daily_Return': ticker})
    if returns_df is None:
        returns_df = df
    else:
        returns_df = pd.merge(returns_df, df, on='Date', how='outer')

# Equal-weighted average return
returns_df['portfolio_return'] = returns_df[TICKERS].mean(axis=1)

# Next-day label (consistent with individual stock logic)
returns_df['label'] = (returns_df['portfolio_return'].shift(-1) > 0).astype(int)
returns_df = returns_df.dropna(subset=['label'])

portfolio = returns_df[['Date', 'portfolio_return', 'label']].copy()

print(f"Portfolio trading days: {len(portfolio)}")
print(f"Portfolio baseline (full period): {portfolio['label'].mean():.1%}")

Portfolio trading days: 1254
Portfolio baseline (full period): 54.1%


In [38]:
# ============================================================
# 5-Stock Fusion (Majority Vote) — Train 2021-2024 only
# ============================================================

from collections import defaultdict

# 1. Select valid signals trained on 2021-2024
valid_2024 = results_df[
    (results_df['train'] == '2021-2024') & 
    (results_df['beat'] == True)
].copy()

print(f"Valid signal combinations (2021-2024): {len(valid_2024)}")
print(valid_2024[['ticker', 'cat', 'beta', 'acc', 'baseline', 'n']].to_string(index=False))

# 2. Collect predictions per date for portfolio and per stock
port_preds = defaultdict(list)                     # date -> list of predictions (all signals)
stock_preds = {ticker: defaultdict(list) for ticker in TICKERS}  # ticker -> date -> list of {pred, label}

for _, row in valid_2024.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    
    key = (ticker, cat)
    if key not in daily_sentiment:
        continue
    daily = daily_sentiment[key].copy()
    
    # Merge with stock labels (labels are already next_label)
    label_df = stock_labels[ticker][['Date', 'label', 'Daily_Return']].copy()
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df.sort_values('Date')
    
    # Test set: 2025 only, exclude days with zero sentiment (align with original filter)
    test = df[(df['Date'] >= '2025-01-01') & (df['sentiment'] != 0)].copy()
    if len(test) == 0:
        continue
    
    test['pred'] = (beta * test['sentiment'] > 0).astype(int)
    
    for _, r in test.iterrows():
        date = r['Date']
        pred = int(r['pred'])
        label = int(r['label'])
        
        # Portfolio-level (all signals pooled)
        port_preds[date].append(pred)
        
        # Per-stock
        stock_preds[ticker][date].append({'pred': pred, 'label': label})

# 3. Portfolio majority vote
port_results = []
for date, preds in sorted(port_preds.items()):
    port_row = portfolio[portfolio['Date'] == date]
    if len(port_row) == 0:
        continue
    port_label = int(port_row['label'].iloc[0])
    
    majority = 1 if sum(preds) >= len(preds) / 2 else 0
    port_results.append({
        'Date': date,
        'label': port_label,
        'pred': majority,
        'n_signals': len(preds),
        'correct': int(majority == port_label)
    })

port_df = pd.DataFrame(port_results)

print("\n=== Portfolio (5-stock fusion, 2021-2024 train) ===")
if len(port_df) > 0:
    acc = port_df['correct'].mean()
    base = port_df['label'].mean()
    print(f"Signal days       : {len(port_df)} / 249")
    print(f"Accuracy          : {acc:.1%}")
    print(f"Baseline (signal) : {base:.1%}")
    print(f"Beat baseline     : {'Yes' if acc > base else 'No'}")
else:
    print("No portfolio prediction days.")

# 4. Individual stock majority vote
print("\n=== Individual stocks (5-stock fusion, 2021-2024 train) ===")
print(f"{'Ticker':<8} {'Days':<8} {'Correct':<10} {'Accuracy':<12} {'Baseline':<12} {'Beat'}")
print('-' * 68)

for ticker in TICKERS:
    ticker_results = []
    for date, signals in stock_preds[ticker].items():
        preds = [s['pred'] for s in signals]
        label = signals[0]['label']   # all signals for same stock/date share the same label
        majority = 1 if sum(preds) >= len(preds) / 2 else 0
        ticker_results.append({
            'Date': date,
            'label': label,
            'pred': majority,
            'n_signals': len(preds),
            'correct': int(majority == label)
        })
    
    df_ticker = pd.DataFrame(ticker_results)
    if len(df_ticker) == 0:
        print(f"{ticker:<8} {'No signal days':<18}")
        continue
    
    acc = df_ticker['correct'].mean()
    base = df_ticker['label'].mean()
    beat = 'Yes' if (acc > base and acc > 0.5) else 'No'
    print(f"{ticker:<8} {len(df_ticker):<8} {df_ticker['correct'].sum():<10} {acc:.1%}        {base:.1%}        {beat}")

# 5. Signal strength distribution (number of concurrent signals per day)
if len(port_df) > 0:
    print("\n=== Signal strength distribution (concurrent signals per day) ===")
    print(port_df['n_signals'].value_counts().sort_index().head(10))

Valid signal combinations (2021-2024): 9
ticker        cat      beta      acc  baseline   n
  NVDA       chip  0.004220 0.603175  0.476190 126
  NVDA regulation  0.002058 0.590909  0.431818  44
  MSFT       chip  0.005536 0.516129  0.451613  31
  MSFT      power  0.003788 0.666667  0.494253  87
  AMZN      power  0.006886 0.580645  0.516129  62
  AMZN regulation -0.000067 0.714286  0.500000  14
  META   earnings  0.019958 0.571429  0.523810  42
 GOOGL      power  0.012624 0.590164  0.524590  61
 GOOGL regulation  0.000407 0.512821  0.435897  39

=== Portfolio (5-stock fusion, 2021-2024 train) ===
Signal days       : 219 / 249
Accuracy          : 61.2%
Baseline (signal) : 54.8%
Beat baseline     : Yes

=== Individual stocks (5-stock fusion, 2021-2024 train) ===
Ticker   Days     Correct    Accuracy     Baseline     Beat
--------------------------------------------------------------------
NVDA     128      69         53.9%        48.4%        Yes
MSFT     100      61         61.0%       

In [44]:
# ============================================================
# Bootstrap Significance Test — Final 5-Stock Weighted Model
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score

def bootstrap_accuracy(y_true, y_pred, n_boot=5000, ci=90):
    accs = []
    n = len(y_true)
    for _ in range(n_boot):
        idx = np.random.choice(n, n, replace=True)
        accs.append(accuracy_score(y_true[idx], y_pred[idx]))
    lo = (100 - ci) / 2
    hi = 100 - lo
    return np.percentile(accs, lo), np.percentile(accs, hi)

# Use the final 5-stock weighted vote results (port_df_w from earlier)
y_true = port_df_w['label'].values
y_pred = port_df_w['pred'].values

acc_obs = accuracy_score(y_true, y_pred)
baseline = y_true.mean()
ci_lo, ci_hi = bootstrap_accuracy(y_true, y_pred, n_boot=5000, ci=90)

print("=== Final Model: 5-Stock Weighted Vote (2021-2024 → 2025) ===")
print(f"Signal days          : {len(port_df_w)} / 249 ({len(port_df_w)/249:.1%} coverage)")
print(f"Observed accuracy    : {acc_obs:.1%}")
print(f"Signal-day baseline  : {baseline:.1%}")
print(f"90% Confidence Interval: [{ci_lo:.1%}, {ci_hi:.1%}]")
print(f"Margin of error      : ±{(ci_hi - ci_lo)/2:.1%}")
print(f"Statistically significant (CI lower > baseline)? {'Yes' if ci_lo > baseline else 'No'}")

=== Final Model: 5-Stock Weighted Vote (2021-2024 → 2025) ===
Signal days          : 219 / 249 (88.0% coverage)
Observed accuracy    : 61.6%
Signal-day baseline  : 54.8%
90% Confidence Interval: [56.2%, 67.1%]
Margin of error      : ±5.5%
Statistically significant (CI lower > baseline)? Yes


In [45]:
# ============================================================
# Large Move Days Analysis — Final 5-Stock Weighted Model
# ============================================================

import numpy as np
from sklearn.metrics import accuracy_score

# 1. Prepare 2025 portfolio returns and rolling volatility
port_2025 = portfolio[portfolio['Date'] >= '2025-01-01'].copy()
# Use shift(1) to avoid look-ahead bias (use only information available up to t-1)
port_2025['rolling_std'] = port_2025['portfolio_return'].shift(1).rolling(21).std()

# 2. Define large move days: abs(return) > 1 * rolling_std
large_move_days = port_2025[
    port_2025['portfolio_return'].abs() > port_2025['rolling_std']
].copy()

print(f"Total trading days in 2025: {len(port_2025)}")
print(f"Large move days (abs(return) > 21d rolling std): {len(large_move_days)}")
print(f"Large move coverage: {len(large_move_days)/len(port_2025):.1%}")

# 3. Merge with your final model predictions (port_df_w from 5-stock weighted vote)
# Only keep days where the model actually had a signal
merge_cols = ['Date', 'pred']  # port_df_w contains Date, label, pred, n_signals, etc.
if 'port_df_w' not in locals():
    print("\n⚠️  Warning: 'port_df_w' not found. Please run the weighted vote cell first.")
else:
    merged = pd.merge(
        large_move_days[['Date', 'portfolio_return', 'rolling_std', 'label']],
        port_df_w[['Date', 'pred']],
        on='Date',
        how='inner'  # only days where our model made a prediction
    )
    
    print(f"\nLarge move days WITH model signal: {len(merged)}")
    print(f"Coverage among large move days: {len(merged)/len(large_move_days):.1%}")
    
    if len(merged) > 0:
        # 4. Performance on large move days
        y_true_large = merged['label'].values
        y_pred_large = merged['pred'].values
        acc_large = accuracy_score(y_true_large, y_pred_large)
        base_large = y_true_large.mean()
        
        print("\n=== Performance on Large Move Days ===")
        print(f"Accuracy (large move): {acc_large:.1%}")
        print(f"Baseline (large move): {base_large:.1%}")
        print(f"Beat baseline         : {'Yes' if acc_large > base_large else 'No'}")
        print(f"Improvement vs full sample (61.6%): {acc_large - 0.616:+.1%}")
        
        # 5. Bootstrap significance (large move subset)
        def bootstrap_acc(y_true, y_pred, n_boot=5000, ci=90):
            accs = []
            n = len(y_true)
            for _ in range(n_boot):
                idx = np.random.choice(n, n, replace=True)
                accs.append(accuracy_score(y_true[idx], y_pred[idx]))
            lo = (100 - ci) / 2
            hi = 100 - lo
            return np.percentile(accs, lo), np.percentile(accs, hi)
        
        ci_lo, ci_hi = bootstrap_acc(y_true_large, y_pred_large, n_boot=5000)
        print(f"\n90% Confidence Interval (large move): [{ci_lo:.1%}, {ci_hi:.1%}]")
        print(f"Statistically significant (CI lower > baseline)? {'Yes' if ci_lo > base_large else 'No'}")
        
        # 6. Summary comparison
        print("\n=== Summary Comparison ===")
        print(f"All signal days (n={len(port_df_w)}):  Accuracy 61.6%, Baseline 54.8%")
        print(f"Large move days (n={len(merged)}):    Accuracy {acc_large:.1%}, Baseline {base_large:.1%}")
        print(f"Absolute lift on large move days: {acc_large - 0.616:+.1%}")
    else:
        print("\nNo overlapping days between large move days and model signal days.")

Total trading days in 2025: 249
Large move days (abs(return) > 21d rolling std): 67
Large move coverage: 26.9%

Large move days WITH model signal: 55
Coverage among large move days: 82.1%

=== Performance on Large Move Days ===
Accuracy (large move): 70.9%
Baseline (large move): 65.5%
Beat baseline         : Yes
Improvement vs full sample (61.6%): +9.3%

90% Confidence Interval (large move): [60.0%, 80.0%]
Statistically significant (CI lower > baseline)? No

=== Summary Comparison ===
All signal days (n=219):  Accuracy 61.6%, Baseline 54.8%
Large move days (n=55):    Accuracy 70.9%, Baseline 65.5%
Absolute lift on large move days: +9.3%


In [46]:
# ============================================================
# Core Infrastructure Fusion (NVDA + MSFT + AMZN, only Chip & Power)
# Train 2021-2024 only
# ============================================================

from collections import defaultdict

# Define the core infrastructure universe
CORE_TICKERS = ['NVDA', 'MSFT', 'AMZN']
CORE_CATS = ['chip', 'power']

# 1. Select signals matching the core infrastructure criteria
valid_core = results_df[
    (results_df['train'] == '2021-2024') & 
    (results_df['beat'] == True) &
    (results_df['ticker'].isin(CORE_TICKERS)) &
    (results_df['cat'].isin(CORE_CATS))
].copy()

print(f"Core Infrastructure signal combinations (Chip + Power only): {len(valid_core)}")
print(valid_core[['ticker', 'cat', 'beta', 'acc', 'baseline', 'n']].to_string(index=False))

# 2. Prepare weighted voting (using training accuracy as weights)
core_port_preds = defaultdict(list)  # date -> list of (pred, weight)

for _, row in valid_core.iterrows():
    ticker = row['ticker']
    cat    = row['cat']
    beta   = row['beta']
    weight = row['acc']  # Use train accuracy as voting weight
    
    key = (ticker, cat)
    if key not in daily_sentiment:
        continue
    daily = daily_sentiment[key].copy()
    
    label_df = stock_labels[ticker][['Date', 'label', 'Daily_Return']].copy()
    df = pd.merge(daily, label_df, on='Date', how='inner')
    df = df.sort_values('Date')
    
    test = df[(df['Date'] >= '2025-01-01') & (df['sentiment'] != 0)].copy()
    if len(test) == 0:
        continue
    
    test['pred'] = (beta * test['sentiment'] > 0).astype(int)
    
    for _, r in test.iterrows():
        date = r['Date']
        pred = int(r['pred'])
        core_port_preds[date].append((pred, weight))

# 3. Weighted majority vote for the portfolio
core_results = []
for date, weighted_preds in sorted(core_port_preds.items()):
    port_row = portfolio[portfolio['Date'] == date]
    if len(port_row) == 0:
        continue
    port_label = int(port_row['label'].iloc[0])
    
    total_weight = sum(w for _, w in weighted_preds)
    weight_1 = sum(w for p, w in weighted_preds if p == 1)
    pred = 1 if (weight_1 / total_weight) >= 0.5 else 0
    
    core_results.append({
        'Date': date,
        'label': port_label,
        'pred': pred,
        'n_signals': len(weighted_preds),
        'correct': int(pred == port_label)
    })

core_df = pd.DataFrame(core_results)

print("\n=== Portfolio (Core Infrastructure: Chip + Power only) ===")
if len(core_df) > 0:
    acc_core = core_df['correct'].mean()
    base_core = core_df['label'].mean()
    print(f"Signal days       : {len(core_df)} / 249")
    print(f"Accuracy (weighted): {acc_core:.1%}")
    print(f"Baseline (signal) : {base_core:.1%}")
    print(f"Beat baseline     : {'Yes' if acc_core > base_core else 'No'}")
    print(f"Improvement vs full 5-stock weighted (61.6%): {acc_core - 0.616:+.1%}")
else:
    print("No portfolio prediction days.")

# 4. Signal distribution
if len(core_df) > 0:
    print("\n=== Signal strength distribution (Core Infrastructure) ===")
    print(core_df['n_signals'].value_counts().sort_index().head(10))

Core Infrastructure signal combinations (Chip + Power only): 4
ticker   cat     beta      acc  baseline   n
  NVDA  chip 0.004220 0.603175  0.476190 126
  MSFT  chip 0.005536 0.516129  0.451613  31
  MSFT power 0.003788 0.666667  0.494253  87
  AMZN power 0.006886 0.580645  0.516129  62

=== Portfolio (Core Infrastructure: Chip + Power only) ===
Signal days       : 194 / 249
Accuracy (weighted): 60.3%
Baseline (signal) : 55.7%
Beat baseline     : Yes
Improvement vs full 5-stock weighted (61.6%): -1.3%

=== Signal strength distribution (Core Infrastructure) ===
n_signals
1    115
2     49
3     23
4      7
Name: count, dtype: int64
